# TD 04 - Classification NLP (Deep Learning)

## Objectif du TD
Construire une pipeline DL simple pour classifier des textes.

Probleme a resoudre: predire la classe d'un resume medical.

Pipeline vise:
1) tokenization + sequence padding,
2) modele neural (Embedding + LSTM/CNN),
3) entrainement,
4) evaluation avec accuracy et F1.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

## Etape 1 - Dataset

Dataset impose: `TimSchopf/medical_abstracts`.

A faire:
- charger les donnees,
- choisir un sous-ensemble pour accelerer,
- separer texte / label.

In [2]:
from datasets import load_dataset

# 1) Chargement du dataset
ds = load_dataset("TimSchopf/medical_abstracts")
TEXT_COL = "medical_abstract"
LABEL_COL = "condition_label"

# On limite la taille pour garder un TD rapide a executer
N = min(10000, len(ds["train"]))
texts = list(ds["train"][TEXT_COL][:N])
labels_raw = np.array(ds["train"][LABEL_COL][:N], dtype=np.int32)

print("Nombre d'exemples utilises:", N)
print("Nombre de classes:", len(set(labels_raw)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Nombre d'exemples utilises: 10000
Nombre de classes: 5


## Etape 2 - Split et encodage des labels

A faire:
- split train/validation,
- adapter les labels en IDs.

### Pourquoi transformer les labels `1..5` en `0..4` ?

Dans ce TD DL (Keras), on utilise une couche de sortie `softmax` avec la perte `sparse_categorical_crossentropy`.
Cette configuration attend des classes encodees en entiers de `0` a `K-1`.

Si les labels du dataset sont `1..5`, on fait simplement `y = y_raw - 1` pour obtenir `0..4`.
C'est suffisant ici, donc pas besoin de `LabelEncoder`.

Pour le texte, on garde volontairement un pretraitement minimal car `TextVectorization` gere deja la normalisation
de base (minuscules et ponctuation) et la tokenization.

In [3]:
from sklearn.model_selection import train_test_split

# 2) Remapping des labels: 1..5 -> 0..4
# Keras (sparse_categorical_crossentropy) attend des classes de 0 a K-1
y = labels_raw - 1
class_names = [str(v) for v in sorted(np.unique(labels_raw))]

X_train, X_val, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

num_classes = len(class_names)
print("Train:", len(X_train), "| Validation:", len(X_val))
print("Classes originales:", class_names)

Train: 8000 | Validation: 2000
Classes originales: ['1', '2', '3', '4', '5']


## Etape 3 - Tokenization / Vectorization

Ici on utilise `TextVectorization` de Keras:
- creation du vocabulaire,
- transformation texte -> sequence d'ids,
- padding automatique a une longueur fixe.

In [4]:
import tensorflow as tf
from tensorflow.keras import layers

# 3) Tokenization / vectorization
max_tokens = 20000
seq_len = 180

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len,
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))

# Conversion explicite vers tf.string pour eviter les erreurs de dtype (ex: <U..., str127968)
X_train_tf = tf.constant(list(X_train), dtype=tf.string)
X_val_tf = tf.constant(list(X_val), dtype=tf.string)
y_train_arr = np.asarray(y_train, dtype=np.int32)
y_val_arr = np.asarray(y_val, dtype=np.int32)


## Etape 4 - Construire et entrainer un modele DL

A faire:
- modele 1: Embedding + LSTM,
- (optionnel) modele 2: Embedding + CNN,
- entrainer quelques epochs.

In [5]:
# 4) Architectures
def build_model(kind="lstm"):
    inp = keras.Input(shape=(), dtype=tf.string)
    x = vectorizer(inp)
    x = layers.Embedding(vectorizer.vocabulary_size(), 64, mask_zero=True)(x)

    if kind == "lstm":
        x = layers.LSTM(64, dropout=0.3, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    elif kind == "cnn":
        x = layers.Conv1D(128, 5, activation="relu")(x)
        x = layers.GlobalMaxPooling1D()(x)
    else:
        raise ValueError("kind doit etre 'lstm' ou 'cnn'")

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

# Training separe de l'evaluation
trained_models = {}
for kind in ["lstm", "cnn"]:
    print("\n" + "=" * 80)
    print("Training:", kind.upper())
    print("=" * 80)

    model = build_model(kind)
    early_stopping_callback = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
    model.fit(
        X_train_tf,
        y_train_arr,
        validation_data=(X_val_tf, y_val_arr),
        epochs=20,
        batch_size=64,
        callbacks=[early_stopping_callback],
        verbose=1,
    )
    trained_models[kind] = model

print("\nTraining termine. Passer a la cellule d'evaluation.")


Training: LSTM
Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3175 - loss: 1.5753 - val_accuracy: 0.2360 - val_loss: 1.5549
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.3420 - loss: 1.4963 - val_accuracy: 0.4270 - val_loss: 1.3797
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.4398 - loss: 1.3650 - val_accuracy: 0.4915 - val_loss: 1.2472
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.4683 - loss: 1.2449 - val_accuracy: 0.4790 - val_loss: 1.3219
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.5316 - loss: 1.1268 - val_accuracy: 0.4760 - val_loss: 1.3733
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6016 - loss: 0.9919 - val_accuracy: 0.4575 - val_loss: 1.3952

Training: CNN
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.3382 - loss: 1.5474 - val_accuracy: 0.5070 - val_loss: 1.3495
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5047 - loss: 1.2931 - val_accuracy: 0.5715 - val_loss: 1.0919
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5603 - loss: 1.0987 - val_accuracy: 0.5960 - val_loss: 0.9884
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.6079 - loss: 0.9339 - val_accuracy: 0.6100 - val_loss: 0.9498
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6775 - loss: 0.8071 - val_accuracy: 0.5855 - val_loss: 0.9831
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7294 - loss: 0.6883 - val_accuracy: 0.5760 - val_loss: 1.0620
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7666 - loss: 0.6072 - val_accuracy: 0.5605 - val_loss: 1.1396

Training termine. Passer a la cellule d'evaluation.


## Etape 5 - Evaluation

Metriques:
- accuracy,
- F1 macro,
- classification report.

Comparer rapidement les architectures testees.

In [6]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

rows = []
reports = {}

for kind, model in trained_models.items():
    pred_proba = model.predict(X_val_tf, verbose=0)
    pred = pred_proba.argmax(axis=1)

    acc = accuracy_score(y_val_arr, pred)
    f1 = f1_score(y_val_arr, pred, average="macro")

    rep = classification_report(
        y_val_arr,
        pred,
        labels=list(range(num_classes)),
        target_names=class_names,
        digits=4,
        zero_division=0,
    )

    rows.append({
        "Modele": kind.upper(),
        "Accuracy": round(acc, 4),
        "F1_macro": round(f1, 4),
    })
    reports[kind] = rep

df = pd.DataFrame(rows).sort_values("F1_macro", ascending=False)
display(df)

best_kind = df.iloc[0]["Modele"].lower()
print("\nClassification report du meilleur modele:")
print(reports[best_kind])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


,Modele,Accuracy,F1_macro
1,CNN,0.6100,0.5890
0,LSTM,0.4915,0.3964



Classification report du meilleur modele:
              precision    recall  f1-score   support

           1     0.6744    0.8575    0.7550       442
           2     0.5362    0.6207    0.5753       203
           3     0.6000    0.3182    0.4158       264
           4     0.6494    0.8255    0.7269       424
           5     0.5363    0.4213    0.4719       667

    accuracy                         0.6100      2000
   macro avg     0.5992    0.6086    0.5890      2000
weighted avg     0.5992    0.6100    0.5916      2000

